In [1]:
from __future__ import annotations

from typing import TYPE_CHECKING, Optional

import pandas as pd

from src.config.secrets import HF_TOKEN_ENV_VAR, require_env
from src.ml.config.loader import load_experiment_config
from src.ml.config.schema import ExperimentConfig, LLMTrainingConfig
from src.ml.datasets.splits import Splits, build_splits
from src.ml.llm.serialization import load_prompt_config, serialize_frame
from src.utils.logging_config import get_logger

if TYPE_CHECKING:
    from datasets import Dataset, DatasetDict

logger = get_logger(__name__)

# HuggingFace split name -> attribute prefix on Splits
SPLIT_NAMES = {"train": "X_train", "validation": "X_val", "test": "X_test"}

# The completion the model learns to produce. Read from ``games_enriched`` — the
# untouched source frame — rather than from the feature frame or metadata:
# margin is an intermediate column deliberately kept out of both, and sourcing
# it here means the LLM needs no change to the shared experiment config.
TARGET_COLUMN = "margin"


In [2]:
from src.ml.llm.dataset import _format_completion, build_split_dataset

from src.config.paths import DEFAULT_TRAIN_LLM_CONFIG_PATH
from src.ml.config.loader import load_experiment_config, load_llm_training_config

In [3]:
LLM_CONFIG = "qlora/llama31_8b/baseline"
args_config = f"configs/train_llm/{LLM_CONFIG}.yaml"

llm_config = load_llm_training_config(args_config)

In [4]:
llm_config.data.source_experiment_config

'configs/train/llm_features.yaml'

In [5]:
from datasets import DatasetDict

experiment_config: Optional[ExperimentConfig] = None
if experiment_config is None:
    experiment_config = load_experiment_config(llm_config.data.source_experiment_config)


In [6]:
logger.info(
    f"Building LLM dataset from {llm_config.data.source_experiment_config} "
    f"(format={llm_config.data.serialization_format})"
)
splits = build_splits(experiment_config)

prompt_config = None
if llm_config.data.serialization_format == "per_team":
    prompt_config = load_prompt_config(llm_config.data.prompt_config)


In [7]:
dataset = DatasetDict(
    {
        name: build_split_dataset(splits, name, llm_config.data.serialization_format, prompt_config)
        for name in SPLIT_NAMES
    }
)

In [8]:
dataset_train = dataset["train"]


In [9]:
dataset_train

Dataset({
    features: ['game_id', 'prompt', 'completion', 'margin', 'margin_scale'],
    num_rows: 23247
})

In [34]:
dataset_train[2]

{'game_id': 29600232,
 'prompt': "Predict the final NBA score margin (home − away) from pre-game stats.\nAdj Win% = win percentage adjusted for strength of schedule, in percent. Ln = last n games.\nScaled Margin = team's avg point differential in hundredths of the season's margin standard deviation, so 100 = one std (positive = outscored opponents).\n\nTeam           Home  Away\nStreak           +1    -3\nVenue streak     +2    -4\nDays at venue     3     3\nRest days         1     1\nBack-to-back      0     0\n\nAdj Win%  L3  L5  L8  L13  L21\nHome      65  50  45   38   36\nAway       0  22  15   34   31\n\nScaled Margin   L3   L5   L8  L13  L21\nHome            20  -24  -13  -25  -30\nAway           -43  -72  -85  -61  -46\n\nAdj Venue win%  L5  L10\nHome            49   42\nAway            22   39\n\nScaled Margin Venue   L5  L10\nHome                  11  -16\nAway                 -86  -56\n\nAvg km/day   1d   3d   7d  14d\nHome          0  211  196  412\nAway        381  466  455

In [13]:
dataset_validation = dataset["validation"]


In [14]:
dataset_validation

Dataset({
    features: ['game_id', 'prompt', 'completion', 'margin', 'margin_scale'],
    num_rows: 2949
})

In [10]:
dataset_test = dataset["test"]

In [12]:
dataset_test


Dataset({
    features: ['game_id', 'prompt', 'completion', 'margin', 'margin_scale'],
    num_rows: 1018
})

In [11]:
dataset_test[0]

{'game_id': 22400030,
 'prompt': "Predict the final NBA score margin (home − away) from pre-game stats.\nAdj Win% = win percentage adjusted for strength of schedule, in percent. Ln = last n games.\nScaled Margin = team's avg point differential in hundredths of the season's margin standard deviation, so 100 = one std (positive = outscored opponents).\n\nTeam                        Home  Away\nStreak                        -1    -1\nVenue streak                  -4    -1\nDays at venue                  1     6\nRest days                      1     1\nBack-to-back                   0     0\nAdj Last-season venue win%    48    36\n\nAdj Win%  L3  L5  L8  L13  L21\nHome      33  49  46   43   40\nAway      38  64  51   35   41\n\nScaled Margin   L3   L5   L8  L13  L21\nHome           -89  -75  -55  -49  -47\nAway           -55  -17  -17  -29  -29\n\nAdj Venue win%  L5  L10\nHome            24   19\nAway            44   39\n\nScaled Margin Venue   L5  L10\nHome                 -81  -88\nAway

In [10]:
from src.ml.training.data_prep import load_and_validate_data, prepare_data
from src.config.paths import REGULAR_SEASON_GAMES_FEATURES_PATH
from pathlib import Path

data_config = experiment_config.data
split_config = experiment_config.splitting
feat_eng_config = experiment_config.feature_engineering
filters_config = experiment_config.filters

data_path = Path(data_config.path) if data_config.path else Path(REGULAR_SEASON_GAMES_FEATURES_PATH)
target_column = data_config.target_column
date_column = data_config.date_column

In [11]:
keepcols = [
    "gameId",
    # "season",
    "gameDateOnlyStr",
    "hometeamPrename",
    "awayteamPrename",
    # "distance_L1_HT",
    # "distance_L3_HT",
    # "distance_L7_HT",
    # "distance_L14_HT",
    # "distance_L1_VT",
    # "distance_L3_VT",
    # "distance_L7_VT",
    # "distance_L14_VT",
    "margin",
    "margin_scaled",
    "margin_avg_L1_HT",
    "margin_avg_L3_HT",
    "margin_avg_L5_HT",
    "margin_avg_L8_HT",
    "margin_avg_L1_VT",
    "margin_avg_L3_VT",
    "margin_avg_L5_VT",
    "margin_avg_L8_VT",
    "sos_adj_record_L1_HT",
    "sos_adj_record_L3_HT",
    "sos_adj_record_L5_HT",
    "sos_adj_record_L8_HT",
    "sos_adj_record_L1_VT",
    "sos_adj_record_L3_VT",
    "sos_adj_record_L5_VT",
    "sos_adj_record_L8_VT",
    "games_played_HT",
    "games_played_VT",
    "hometeamConference",
    "awayteamConference",
]

In [12]:
games_enriched = load_and_validate_data(
    data_path=data_path, target_column=target_column, date_column=date_column
)

In [22]:
games_enriched[games_enriched["gameId"] == int(dataset_train.data["game_id"][3])]

,gameId,gameDate,gameDateOnlyStr,gameTime,season,hometeamPrename,hometeamName,hometeamId,awayteamPrename,awayteamName,...,gds_L55_VT,gds_L82_VT,gds_L5_HT_at_home,gds_L10_HT_at_home,gds_L20_HT_at_home,gds_L41_HT_at_home,gds_L5_VT_on_road,gds_L10_VT_on_road,gds_L20_VT_on_road,gds_L41_VT_on_road
31289,29600244,1996-12-05 20:30:00,1996-12-05,20:30,1996/97,Chicago,Bulls,1610612741,Los Angeles,Clippers,...,-0.159182,-0.159182,0.393504,0.32792,0.32792,0.32792,-0.312315,-0.094734,-0.094734,-0.094734


In [15]:
games_enriched.loc[
    ((games_enriched["hometeamName"] == "Lakers") | (games_enriched["awayteamName"] == "Lakers"))
    & (games_enriched["season"] == "1996/97"),
    keepcols,
]


,gameId,gameDateOnlyStr,hometeamPrename,awayteamPrename,margin,margin_scaled,margin_avg_L1_HT,margin_avg_L3_HT,margin_avg_L5_HT,margin_avg_L8_HT,...,sos_adj_record_L5_HT,sos_adj_record_L8_HT,sos_adj_record_L1_VT,sos_adj_record_L3_VT,sos_adj_record_L5_VT,sos_adj_record_L8_VT,games_played_HT,games_played_VT,hometeamConference,awayteamConference
31058,29600012,1996-11-01,Los Angeles,Phoenix,14,1.056362,0.0,0.000000,0.0,0.000,...,NaN,NaN,NaN,NaN,NaN,NaN,0,0,West,West
31074,29600027,1996-11-03,Los Angeles,Minnesota,6,0.450723,14.0,14.000000,14.0,14.000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1,1,West,West
31080,29600031,1996-11-05,New York,Los Angeles,-6,-0.450221,27.0,17.500000,17.5,17.500,...,0.583227,0.583227,1.150376,0.583227,0.583227,0.583227,2,2,East,West
31089,29600044,1996-11-06,Charlotte,Los Angeles,10,0.749439,-27.0,-8.000000,-8.0,-8.000,...,0.582136,0.582136,1.491124,1.034908,1.034908,1.034908,2,3,East,West
31100,29600057,1996-11-08,Toronto,Los Angeles,1,0.074981,4.0,-5.000000,-5.0,-5.000,...,0.407301,0.407301,0.000000,0.928997,0.790419,0.790419,3,4,East,West
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32167,29601121,1997-04-11,Los Angeles,Phoenix,16,1.207601,-12.0,6.333333,6.6,7.625,...,0.476735,0.603272,0.810997,0.993085,0.978094,1.052952,77,77,West,West
32181,29601135,1997-04-13,Los Angeles,Utah,2,0.150560,16.0,9.333333,4.8,6.875,...,0.529428,0.623534,1.367830,1.084539,0.866341,0.807515,78,77,West,West
32209,29601162,1997-04-17,Los Angeles,Sacramento,9,0.677545,2.0,2.000000,7.4,6.625,...,0.872440,0.735485,0.503463,0.498099,0.632541,0.517438,79,79,West,West
32219,29601173,1997-04-18,Los Angeles,Los Angeles,-28,-2.100496,-4.0,8.000000,0.4,5.375,...,0.388148,0.542270,0.830435,1.124316,0.908704,0.682888,80,80,West,West


In [16]:
for col in games_enriched.columns:
    print(col)

gameId
gameDate
gameDateOnlyStr
gameTime
season
hometeamPrename
hometeamName
hometeamId
awayteamPrename
awayteamName
awayteamId
homeScore
awayScore
winner
overtimes
postponed
gameType
hometeamLocation
gameLocation
awayteamLocation
hometeamConference
awayteamConference
winnerteamConference
is_neutral_court_game
neutral_court
win_bool
margin
margin_scaled
total_wins_HT
total_losses_HT
games_played_HT
record_L1_HT
record_L3_HT
record_L5_HT
record_L8_HT
record_L13_HT
record_L21_HT
record_L34_HT
record_L55_HT
record_L82_HT
total_wins_VT
total_losses_VT
games_played_VT
record_L1_VT
record_L3_VT
record_L5_VT
record_L8_VT
record_L13_VT
record_L21_VT
record_L34_VT
record_L55_VT
record_L82_VT
total_wins_HT_at_home
total_losses_HT_at_home
games_played_HT_at_home
record_L5_HT_at_home
record_L10_HT_at_home
record_L20_HT_at_home
record_L41_HT_at_home
total_wins_VT_on_road
total_losses_VT_on_road
games_played_VT_on_road
record_L5_VT_on_road
record_L10_VT_on_road
record_L20_VT_on_road
record_L41_VT_on

In [23]:
game_id = int(dataset_train.data["game_id"][3])
games_enriched.loc[
    (games_enriched["gameId"] == game_id),
    keepcols,
]

,gameId,gameDateOnlyStr,hometeamPrename,awayteamPrename,margin,margin_scaled,margin_avg_L1_HT,margin_avg_L3_HT,margin_avg_L5_HT,margin_avg_L8_HT,...,sos_adj_record_L5_HT,sos_adj_record_L8_HT,sos_adj_record_L1_VT,sos_adj_record_L3_VT,sos_adj_record_L5_VT,sos_adj_record_L8_VT,games_played_HT,games_played_VT,hometeamConference,awayteamConference
31289,29600244,1996-12-05,Chicago,Los Angeles,18,1.339379,3.0,10.333333,6.0,11.25,...,0.74437,0.721824,1.012232,0.363451,0.27056,0.270877,17,17,East,West


In [24]:
dataset_train.data["prompt"][3]

<pyarrow.StringScalar: "Predict the final NBA score margin (home − away) from pre-game stats.\nAdj Win% = win percentage adjusted for strength of schedule, in percent. Ln = last n games.\nScaled Margin = team's avg point differential in hundredths of the season's margin standard deviation, so 100 = one std (positive = outscored opponents).\n\nTeam                        Home  Away\nStreak                        +4    +1\nVenue streak                  +6    -3\nDays at venue                  1     1\nRest days                      1     1\nBack-to-back                   0     0\nAdj Last-season venue win%    92    25\n\nAdj Win%  L3  L5  L8  L13  L21\nHome      68  74  72   82   82\nAway      36  27  27   40   43\n\nScaled Margin   L3   L5   L8  L13  L21\nHome            77   45   84  112  115\nAway           -52  -61  -39  -22  -15\n\nAdj Venue win%  L5  L10\nHome            92   92\nAway            27   55\n\nScaled Margin Venue   L5  L10\nHome                 170  178\nAway          

In [25]:
print(str(dataset_train.data["prompt"][3]))

Predict the final NBA score margin (home − away) from pre-game stats.
Adj Win% = win percentage adjusted for strength of schedule, in percent. Ln = last n games.
Scaled Margin = team's avg point differential in hundredths of the season's margin standard deviation, so 100 = one std (positive = outscored opponents).

Team                        Home  Away
Streak                        +4    +1
Venue streak                  +6    -3
Days at venue                  1     1
Rest days                      1     1
Back-to-back                   0     0
Adj Last-season venue win%    92    25

Adj Win%  L3  L5  L8  L13  L21
Home      68  74  72   82   82
Away      36  27  27   40   43

Scaled Margin   L3   L5   L8  L13  L21
Home            77   45   84  112  115
Away           -52  -61  -39  -22  -15

Adj Venue win%  L5  L10
Home            92   92
Away            27   55

Scaled Margin Venue   L5  L10
Home                 170  178
Away                 -37   -2

Avg km/day    1d   3d   7d  14d
H

In [ ]:
from pathlib import Path

from datasets import DatasetDict

from src.config.paths import DEFAULT_TRAIN_LLM_CONFIG_PATH
from src.ml.config.loader import load_experiment_config, load_llm_training_config
from src.ml.datasets.splits import Splits, build_splits
from src.ml.llm.dataset import (
    SPLIT_NAMES,
    build_llm_dataset,
    build_split_dataset,
    push_to_hub,
    summarize,
)

In [ ]:
LLM_CONFIG = "qlora/llama31_8b/baseline"
args_config = f"configs/train_llm/{LLM_CONFIG}.yaml"

In [ ]:
DEFAULT_TRAIN_LLM_CONFIG_PATH

In [ ]:
llm_config = load_llm_training_config(args_config)

In [ ]:
llm_config

In [ ]:
experiment_config = None
if experiment_config is None:
    experiment_config = load_experiment_config(llm_config.data.source_experiment_config)


In [ ]:
experiment_config

In [ ]:
splits = build_splits(experiment_config)

In [ ]:
llm_config.data.serialization_format

In [ ]:
dataset = DatasetDict(
    {
        name: build_split_dataset(splits, name, llm_config.data.serialization_format)
        for name in SPLIT_NAMES
    }
)

In [ ]:
# push_to_hub(dataset, llm_config.data.dataset_name, private=llm_config.hub.private)


In [ ]:
dataset_train = dataset["train"]


In [ ]:
dataset_train

In [ ]:
print(dataset_train.data["completion"][203])


In [ ]:
print(dataset_train.data["prompt"][203])

In [ ]:
print(dataset_train.data["completion"][1])


In [ ]:
dataset_test = dataset["test"]

In [ ]:
print(dataset_test.data["prompt"][1])

In [ ]:
for i in range(90, min(90 + 32, 100)):
    print(i)

In [ ]:
WARNING:bitsandbytes.autograd._functions:MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization